# Modelagem e o ciclo da computação científica

[Abrir este capítulo no Google Colab](https://colab.research.google.com/github/vnikoofard/calculo-numerico-livro/blob/main/livro/01-modelagem.ipynb)

> **Objetivos.** Ao final deste capítulo você será capaz de:
> 1. transformar uma lei física numa equação diferencial e numa fórmula de recorrência que o computador resolve;
> 2. distinguir **verificação** (o código resolve corretamente as equações do modelo?) de **validação** (o modelo descreve a realidade?);
> 3. medir a ordem de convergência de uma discretização simples;
> 4. reconhecer quando o passo de tempo é grande demais.

In [ ]:
# Preparação do ambiente: funciona no seu computador, no Colab e no navegador (JupyterLite).
import sys
try:
    import cnlivro
except ImportError:
    if sys.platform == "emscripten":  # JupyterLite: Python rodando no navegador
        import micropip
        await micropip.install(["plotly", "scipy",
            "https://vnikoofard.github.io/calculo-numerico-livro/pacote/cnlivro-0.1.0-py3-none-any.whl"])
    else:  # Colab ou instalação local sem o pacote
        %pip install -q git+https://github.com/vnikoofard/calculo-numerico-livro

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from cnlivro.estilo import aplicar_estilo, mostrar, grafico_convergencia, grafico_iteracoes, com_controle, numero, CORES
from cnlivro import raizes
from cnlivro.verificacao import ordem_observada, ordem_iterativa, compara

aplicar_estilo()

## 1. Problema de engenharia: o salto de paraquedas

Um paraquedista de massa $m = 68{,}1$ kg salta de um balão parado. Durante a queda livre, a resistência do ar cresce com a velocidade. Duas perguntas práticas:

- Qual a velocidade máxima que ele atinge (a **velocidade terminal**)?
- Quanto tempo leva para chegar perto dela?

Este problema clássico acompanha o livro inteiro, porque contém em miniatura todas as etapas da computação científica: modelar, discretizar, resolver, verificar e validar.

## 2. A ideia em uma imagem

Um modelo matemático transforma uma pergunta física numa equação. Quando a equação não tem solução exata, ou é cara demais de resolver exatamente, nós a **discretizamos**: trocamos o tempo contínuo por passos de tamanho $\Delta t$ e a derivada por uma diferença. A figura compara a solução exata com a solução discreta, que avança em linha reta durante cada passo.

In [ ]:
g, c, m = 9.81, 12.5, 68.1        # m/s², kg/s, kg

def v_exata(t):
    return g * m / c * (1 - np.exp(-c / m * t))

def euler_paraquedista(dt, t_final=30.0, v0=0.0):
    n = int(round(t_final / dt))
    t = np.linspace(0, n * dt, n + 1)
    v = np.empty(n + 1)
    v[0] = v0
    for i in range(n):
        v[i + 1] = v[i] + (g - c / m * v[i]) * dt
    return t, v

In [ ]:
def figura_euler(dt, titulo=None):
    t_fino = np.linspace(0, 30, 300)
    t, v = euler_paraquedista(dt)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=t_fino, y=v_exata(t_fino), name="solução exata", line=dict(color=CORES["exata"])))
    fig.add_trace(go.Scatter(x=t, y=v, name=f"Euler, Δt = {numero(dt)} s", mode="lines+markers",
                             line=dict(color=CORES["aproximacao"])))
    fig.add_hline(y=g * m / c, line=dict(color=CORES["referencia"], dash="dash", width=1),
                  annotation_text="velocidade terminal", annotation_position="bottom right")
    fig.update_xaxes(title="tempo t (s)", range=[0, 30])
    fig.update_yaxes(title="velocidade v (m/s)", range=[0, 60])
    fig.update_layout(title=titulo or f"Euler com Δt = {numero(dt)} s")
    return fig

mostrar(figura_euler(2.0))

**O que notar:** em cada passo, a solução discreta segue a reta tangente à curva no início do passo. Por isso ela fica sempre acima da curva exata, mas ambas tendem à mesma velocidade terminal, cerca de 53,4 m/s (192 km/h).

## 3. Formulação matemática

### 3.1 O modelo

Pela segunda lei de Newton, $m\,\dfrac{dv}{dt} = F_{\text{peso}} - F_{\text{arrasto}}$. Supondo arrasto **proporcional à velocidade**, $F_{\text{arrasto}} = c\,v$, em que $c$ é o **coeficiente de arrasto linear**, medido em kg/s (força dividida por velocidade: N/(m/s) = kg/s):

$$
\frac{dv}{dt} = g - \frac{c}{m}\, v .
$$ (eq-paraquedista)

Partindo do repouso, $v(0) = 0$, a solução exata é

$$
v(t) = \frac{g m}{c}\left(1 - e^{-(c/m)\,t}\right).
$$ (eq-paraquedista-exata)

Ao longo do capítulo usamos os valores

| grandeza | símbolo | valor |
| --- | --- | --- |
| massa do paraquedista | $m$ | 68,1 kg |
| coeficiente de arrasto linear | $c$ | 12,5 kg/s |
| aceleração da gravidade | $g$ | 9,81 m/s² |

O valor de $c$ não foi medido: ele foi escolhido para que a velocidade terminal fique próxima da de um paraquedista real. A seção 7.2 volta a essa escolha.

Quando $t \to \infty$, a exponencial some e $v \to v_T = gm/c$. Na velocidade terminal, o arrasto equilibra o peso. Com os valores da tabela, $v_T = 9{,}81 \times 68{,}1 / 12{,}5 \approx 53{,}4$ m/s.

**Quão rápido a velocidade chega lá?** A combinação $\tau = m/c$ tem unidade de tempo (kg ÷ kg/s = s) e se chama **constante de tempo**. Com ela, a solução {eq}`eq-paraquedista-exata` fica numa forma que não depende dos valores particulares do problema:

$$
\frac{v(t)}{v_T} = 1 - e^{-t/\tau}.
$$

O termo $e^{-t/\tau}$ é a fração que ainda **falta** para chegar a $v_T$. A cada intervalo de duração $\tau$, essa fração é multiplicada por $e^{-1} \approx 0{,}37$, ou seja, cai para cerca de um terço do que era:

| instante $t$ | fração atingida $1 - e^{-t/\tau}$ | fração que falta $e^{-t/\tau}$ |
| --- | --- | --- |
| $\tau$ | 63,2 % | 36,8 % |
| $2\tau$ | 86,5 % | 13,5 % |
| $3\tau$ | 95,0 % | 5,0 % |
| $5\tau$ | 99,3 % | 0,7 % |

A velocidade **nunca** chega exatamente a $v_T$, porque a exponencial só tende a zero. Para dizer "já chegou", é preciso escolher uma tolerância. Uma convenção comum em engenharia é a de 5 %, atingida em $3\tau$, porque $e^{-3} \approx 0{,}050$. Para uma tolerância de 1 %, usa-se $5\tau$. No nosso salto, $\tau = 68{,}1/12{,}5 \approx 5{,}4$ s. Depois de $3\tau \approx 16$ s, a velocidade já está a 95 % de $v_T$, cerca de 50,8 m/s. Isso responde à segunda pergunta da seção 1.

In [ ]:
tau = m / c
vT = g * m / c
t_fino = np.linspace(0, 6 * tau, 300)
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_fino / tau, y=100 * v_exata(t_fino) / vT, showlegend=False,
                         line=dict(color=CORES["exata"])))
for k, posicao in [(1, "bottom right"), (3, "bottom right"), (5, "bottom left")]:
    frac = 100 * (1 - np.exp(-k))
    fig.add_trace(go.Scatter(x=[k, k, 0], y=[0, frac, frac], mode="lines", showlegend=False,
                             line=dict(color=CORES["referencia"], dash="dot", width=1)))
    fig.add_trace(go.Scatter(x=[k], y=[frac], mode="markers+text", showlegend=False,
                             text=[f"{k if k > 1 else ''}τ ≈ {numero(k * tau, 1)} s: {numero(frac, 1)} %"],
                             textposition=posicao, marker=dict(color=CORES["aproximacao"], size=9)))
fig.update_xaxes(title="tempo em unidades de τ (t / τ)", range=[0, 6])
fig.update_yaxes(title="porcentagem da velocidade terminal (%)", range=[0, 105])
fig.update_layout(title=f"A cada τ ≈ {numero(tau, 1)} s, o que falta até a velocidade terminal cai para 37 %")
mostrar(fig)

**O que notar:** no eixo horizontal o tempo está medido em unidades de $\tau$, por isso a curva serve para qualquer massa e qualquer coeficiente $c$. Os patamares 63 %, 95 % e 99 % aparecem sempre em $\tau$, $3\tau$ e $5\tau$. Mudar $m$ ou $c$ só muda quantos segundos vale cada $\tau$.

Todo modelo tem **hipóteses**, e esta tem várias: arrasto linear, $g$ constante, densidade do ar constante, queda vertical. Cada uma delas pode ser questionada, e a seção 7 mostra o que acontece quando uma falha.

### 3.2 A discretização

A derivada é o limite de uma razão de diferenças:

$$
\frac{dv}{dt} = \lim_{\Delta t \to 0} \frac{v(t + \Delta t) - v(t)}{\Delta t}.
$$

Usando a razão com $\Delta t$ **finito**, a equação {eq}`eq-paraquedista` vira

$$
\frac{v_{i+1} - v_i}{\Delta t} \approx g - \frac{c}{m}\, v_i
\quad\Longrightarrow\quad
v_{i+1} = v_i + \left(g - \frac{c}{m}\, v_i\right)\Delta t .
$$ (eq-euler)

Esse é o **método de Euler**: novo valor = valor antigo + inclinação × passo. É uma fórmula de recorrência que o computador avalia com somas e multiplicações.

### 3.3 Quanto erramos?

:::{admonition} Revisão rápida: a série de Taylor
:class: tip

**A ideia.** Se conhecemos uma função suave $f$ num ponto $t$ (o valor, a inclinação, a curvatura...), conseguimos prever o valor num ponto vizinho $t + h$:

$$
f(t+h) = f(t) + f'(t)\,h + \frac{f''(t)}{2!}\,h^2 + \frac{f'''(t)}{3!}\,h^3 + \cdots
$$

Cada termo corrige o anterior. Com só o primeiro termo, supomos que $f$ é constante. Com os dois primeiros, seguimos a **reta tangente**, que é exatamente o que o método de Euler faz. O terceiro acrescenta a curvatura, e assim por diante.

**Truncar e o resto.** Na prática, paramos num termo. O teorema de Taylor garante que tudo o que foi descartado cabe num único termo, calculado num ponto $\xi$ desconhecido entre $t$ e $t + h$:

$$
f(t+h) = f(t) + f'(t)\,h + \tfrac12 f''(\xi)\,h^2 .
$$

Não sabemos onde está $\xi$, mas sabemos como o resto **escala**: ele é proporcional a $h^2$. Se $h$ cai pela metade, o resto cai para um quarto. É isso que a notação $O(h^2)$ quer dizer.

**Exemplo.** Para $f(t) = e^t$ em torno de $t = 0$ com $h = 0{,}1$, o valor exato é $e^{0{,}1} = 1{,}105171\ldots$

| termos usados | aproximação | erro |
| --- | --- | --- |
| $1$ | 1 | 0,105 |
| $1 + h$ | 1,1 | 0,0052 |
| $1 + h + h^2/2$ | 1,105 | 0,00017 |

Cada termo a mais divide o erro por 20 a 30: cada potência de $h = 0{,}1$ é dez vezes menor que a anterior, e o fatorial no denominador também cresce.

No paraquedista, a função é $f = v$ e o passo é $h = \Delta t$. O capítulo 2 (seção 3.4) volta à série de Taylor com mais detalhes.
:::

Pela série de Taylor, $v(t + \Delta t) = v(t) + v'(t)\,\Delta t + \tfrac12 v''(\xi)\,\Delta t^2$. Euler joga fora o último termo, então comete em **cada passo** um erro da ordem de $\Delta t^2$. Para chegar a um instante fixo $T$ são necessários $T/\Delta t$ passos, e os erros se acumulam:

$$
\text{erro global} \sim \frac{T}{\Delta t}\cdot \Delta t^2 = O(\Delta t).
$$

Euler é um método de **ordem 1**: dividir o passo por 2 divide o erro por 2. Na seção 6 vamos **medir** essa ordem, e o capítulo 12 trata de métodos de ordem mais alta.

### 3.4 O ciclo completo

A computação científica é um ciclo com duas perguntas diferentes:

- **Verificação**: *o código resolve corretamente as equações do modelo?* Ela compara o código com a matemática: solução exata, ordem de convergência, resíduo.
- **Validação**: *o modelo descreve a realidade?* Ela compara o modelo com a realidade: medidas de laboratório, dados de campo.

Em inglês, a distinção costuma ser resumida num jogo de palavras atribuído a P. J. Roache: verificação é *solving the equations right* (resolver as equações do jeito certo) e validação é *solving the right equations* (resolver as equações certas). A primeira pergunta é sobre **matemática e programação**. A segunda é sobre **física**.

Um código pode estar perfeitamente verificado e o modelo, mesmo assim, ser inválido. A seção 7.2 mostra um exemplo. A ordem importa: só faz sentido validar um modelo depois de verificar o código, porque, senão, não se sabe se a discordância vem do modelo ou de um erro no programa.

## 4. À mão

Aplique a fórmula {eq}`eq-euler` com $\Delta t = 2$ s, $c/m = 12{,}5/68{,}1 = 0{,}18355$ s⁻¹ e $v_0 = 0$:

| $i$ | $t_i$ (s) | $v_i$ por Euler (m/s) | $v(t_i)$ exata (m/s) | erro relativo |
| --- | --- | --- | --- | --- |
| 0 | 0 | 0 | 0 | — |
| 1 | 2 | $0 + (9{,}81 - 0)\cdot 2 = 19{,}62$ | 16,42 | 19,5 % |
| 2 | 4 | $19{,}62 + (9{,}81 - 3{,}601)\cdot 2 = 32{,}04$ | 27,80 | 15,3 % |
| 3 | 6 | $32{,}04 + (9{,}81 - 5{,}881)\cdot 2 = 39{,}90$ | 35,68 | 11,8 % |

O erro relativo **diminui** ao longo do tempo. As duas soluções se aproximam da mesma velocidade terminal, e lá o erro desaparece.

In [ ]:
t, v = euler_paraquedista(2.0, t_final=6.0)
for ti, vi in zip(t, v):
    print(f"t = {ti:4.1f} s   Euler = {vi:7.3f}   exata = {v_exata(ti):7.3f} m/s")

## 5. Explore

Arraste o slider para mudar o passo $\Delta t$. Repare que a solução de Euler se aproxima da exata quando $\Delta t$ diminui. Para $\Delta t$ maior que a constante de tempo $\tau = m/c \approx 5{,}4$ s (seção 3.1), os pontos passam a **ultrapassar** a velocidade terminal e oscilar. A seção 7 explica por quê.

In [ ]:
passos = [0.25, 0.5, 1.0, 2.0, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0]
quadros = [(numero(dt), figura_euler(dt)) for dt in passos]
mostrar(com_controle(quadros, rotulo="Δt (s) = ", ativo=7, titulos=[q[1].layout.title.text for q in quadros]))

## 6. Trabalhando com o agente

O modelo linear é bom para aprender, mas o arrasto real em alta velocidade é **quadrático**: $F_{\text{arrasto}} = c_2\, v^2$. Além disso, o paraquedas abre num instante $t_a$, e o coeficiente de arrasto cresce muito. Esse modelo já não tem solução simples à mão. É uma boa tarefa para um agente.

### 6.1 O pedido

> *"Um paraquedista de massa `m` (kg) salta do repouso e cai em queda livre. No instante `t_abertura` (s) ele abre o paraquedas, e o coeficiente de arrasto quadrático $c_2$ passa de `c2_queda_livre` para `c2_paraquedas` (ambos em kg/m). Escreva em Python a função `simular_salto(dt, t_final, t_abertura, c2_queda_livre=0.25, c2_paraquedas=15.0, m=68.1)` que integra $dv/dt = g - (c_2/m)\,v^2$, com $v(0) = 0$, pelo método de Euler com passo `dt` (s), do instante 0 até `t_final` (s). Devolva os vetores de tempo e de velocidade. Escreva testes que verifiquem: (1) antes da abertura, a velocidade tende a $\sqrt{mg/c_2}$; (2) depois da abertura, ela tende ao novo valor terminal; (3) o erro cai pela metade quando `dt` cai pela metade."*

Para não confundir os dois modelos, o coeficiente do arrasto linear continua sendo $c$ ($F = c\,v$, em kg/s), e o do arrasto quadrático se chama $c_2$ ($F = c_2\,v^2$). O índice 2 lembra o expoente de $v$. O coeficiente $c_2$ é medido em kg/m (força dividida por velocidade ao quadrado: N/(m/s)² = kg/m). Ele resume $c_2 = \tfrac12\,\rho\, C_D\, A$, em que $\rho$ é a densidade do ar, $A$ a área frontal e $C_D$ o **coeficiente de arrasto** adimensional, aquele que aparece nas tabelas de aerodinâmica. Não confunda $C_D$, que é um número puro, com $c_2$, que tem unidade. Os valores do pedido, 0,25 kg/m em queda livre e 15 kg/m com o paraquedas aberto, foram escolhidos para dar velocidades terminais de cerca de 52 m/s e 7 m/s, próximas das reais.

O pedido já traz as verificações. Elas usam o que sabemos do modelo mesmo sem resolvê-lo: a velocidade terminal sai de igualar peso e arrasto, $mg = c_2 v_T^2$.

### 6.2 O código que voltou

In [ ]:
# Código devolvido pelo agente (copiado sem alterações)
def simular_salto(dt, t_final, t_abertura, c2_queda_livre=0.25, c2_paraquedas=15.0, m=68.1):
    g = 9.81
    n = int(t_final / dt)
    t = [0.0]
    v = [0.0]
    c2 = c2_queda_livre
    for i in range(n):
        if t[-1] == t_abertura:
            c2 = c2_paraquedas
        v.append(v[-1] + (g - c2 / m * v[-1] ** 2) * dt)
        t.append(t[-1] + dt)
    return np.array(t), np.array(v)

### 6.3 Verificação: as velocidades terminais

In [ ]:
vT_livre = np.sqrt(68.1 * 9.81 / 0.25)     # terminal em queda livre
vT_aberto = np.sqrt(68.1 * 9.81 / 15.0)    # terminal com o paraquedas aberto
saltos = {dt: simular_salto(dt, 60.0, t_abertura=20.0) for dt in (0.0625, 0.1)}
for dt, (t, v) in saltos.items():
    print(f"Δt = {numero(dt)} s:  v(60 s) = {numero(v[-1], 1)} m/s   (esperado: {numero(vT_aberto, 1)} m/s)")

In [ ]:
fig = go.Figure()
for dt, cor, traco in [(0.0625, CORES["secundaria"], "solid"), (0.1, CORES["aproximacao"], "dash")]:
    t, v = saltos[dt]
    fig.add_trace(go.Scatter(x=t, y=v, name=f"código do agente, Δt = {numero(dt)} s",
                             line=dict(color=cor, dash=traco, width=2.5)))
for vt, nome in [(vT_livre, "terminal em queda livre"), (vT_aberto, "terminal com paraquedas")]:
    fig.add_hline(y=vt, line=dict(color=CORES["referencia"], dash="dash", width=1),
                  annotation_text=f"{nome}: {numero(vt, 1)} m/s", annotation_position="top right")
fig.add_vline(x=20, line=dict(color=CORES["erro"], dash="dot", width=1), annotation_text="abertura (t = 20 s)", annotation_position="bottom right")
fig.update_xaxes(title="tempo t (s)")
fig.update_yaxes(title="velocidade v (m/s)")
fig.update_layout(title="Com Δt = 0,1 s o paraquedas nunca abre")
mostrar(fig)

**O que notar:** com $\Delta t = 0{,}0625$ s o paraquedas abre em $t = 20$ s, como esperado. Com $\Delta t = 0{,}1$ s ele **nunca abre**, e o paraquedista continua a 51,7 m/s até o chão.

A causa é o teste `t[-1] == t_abertura`. Somando 0,1 duzentas vezes não se obtém exatamente 20,0, porque 0,1 não tem representação exata em binário (capítulo 2). Já $0{,}0625 = 1/16 = 2^{-4}$ é exato em binário, e a soma também é exata. **O código funciona por sorte para alguns valores de `dt` e falha para outros.** Um teste com um único `dt` não teria encontrado o erro.

Por que não usar $\Delta t = 0{,}25 = 2^{-2}$, que também é exato? Porque, com o paraquedas aberto, o arrasto é 60 vezes maior ($15/0{,}25 = 60$) e o método de Euler fica **instável** com esse passo: a velocidade explode em poucos passos. A seção 7.1 explica esse fenômeno.

In [ ]:
t_soma = 0.0
for _ in range(200):
    t_soma += 0.1
print(f"200 × 0,1 = {t_soma!r}   (igual a 20,0? {t_soma == 20.0})")

A correção é comparar com uma desigualdade (`t[-1] >= t_abertura`) ou, melhor ainda, calcular o tempo como `i * dt` em vez de acumular somas.

### 6.4 Verificação: a ordem de convergência

Com o código corrigido, verificamos a ordem do método. Como não há solução exata para o arrasto quadrático com abertura, usamos como referência uma simulação com passo muito pequeno.

In [ ]:
def simular_salto_corrigido(dt, t_final, t_abertura, c2_queda_livre=0.25, c2_paraquedas=15.0, m=68.1):
    g = 9.81
    n = int(round(t_final / dt))
    t = np.arange(n + 1) * dt
    v = np.zeros(n + 1)
    for i in range(n):
        c2 = c2_paraquedas if t[i] >= t_abertura else c2_queda_livre
        v[i + 1] = v[i] + (g - c2 / m * v[i] ** 2) * dt
    return t, v

T = 15.0     # comparamos as velocidades em t = 15 s (antes da abertura)
_, v_ref = simular_salto_corrigido(1e-5, T, t_abertura=20.0)
dts = np.array([0.4, 0.2, 0.1, 0.05, 0.025, 0.0125])
erros = [abs(simular_salto_corrigido(dt, T, 20.0)[1][-1] - v_ref[-1]) for dt in dts]

mostrar(grafico_convergencia(dts, erros, ordens=(1, 2), nome="erro em t = 15 s",
                             titulo="Euler no salto com arrasto quadrático", rotulo_x="passo Δt (s)",
                             rotulo_y="erro na velocidade (m/s)"))
print("ordem observada:", numero(ordem_observada(dts, erros), 2))

**O que notar:** os pontos ficam paralelos à reta de ordem 1, como previsto na seção 3.3. Esse teste é importante porque verifica o código mesmo sem conhecer a solução exata.

## 7. Quando falha

### 7.1 Passo grande demais: instabilidade

Volte ao slider da seção 5 com $\Delta t = 10$ s. Por que a solução oscila? Reescrevendo {eq}`eq-euler` em termos do desvio $e_i = v_i - v_T$ em relação à velocidade terminal:

$$
e_{i+1} = \left(1 - \frac{c}{m}\,\Delta t\right) e_i .
$$

O desvio é multiplicado pelo fator $\lambda = 1 - (c/m)\,\Delta t$ a cada passo. Há três regimes:

- $0 < \lambda < 1$, ou seja, $\Delta t < m/c \approx 5{,}4$ s: aproximação suave, sem ultrapassar $v_T$;
- $-1 < \lambda < 0$, ou seja, $5{,}4 < \Delta t < 2m/c \approx 10{,}9$ s: oscilação amortecida em torno de $v_T$;
- $\lambda < -1$, ou seja, $\Delta t > 10{,}9$ s: a oscilação **cresce**, e a solução explode.

In [ ]:
fig = go.Figure()
t_fino = np.linspace(0, 120, 400)
fig.add_trace(go.Scatter(x=t_fino, y=v_exata(t_fino), name="exata", line=dict(color=CORES["exata"])))
for dt, cor in [(8.0, CORES["aproximacao"]), (10.0, CORES["secundaria"]), (12.0, CORES["erro"])]:
    t, v = euler_paraquedista(dt, t_final=120.0)
    fig.add_trace(go.Scatter(x=t, y=v, mode="lines+markers", name=f"Δt = {numero(dt)} s", line=dict(color=cor)))
fig.update_xaxes(title="tempo t (s)")
fig.update_yaxes(title="velocidade v (m/s)", range=[-20, 130])
fig.update_layout(title="Acima de Δt ≈ 10,9 s, Euler se torna instável")
mostrar(fig)

**O que notar:** com $\Delta t = 12$ s, a solução numérica explode, embora a solução exata seja perfeitamente calma. A **instabilidade numérica** é uma propriedade do método, não da física. Ela volta em todo o estudo de EDOs (capítulo 13) e EDPs (capítulo 15).

### 7.2 Modelo errado: verificado, mas inválido

O código de Euler da seção 2 está verificado: converge com ordem 1 para a solução exata do modelo linear. Mas o modelo linear prevê $v_T \approx 53{,}4$ m/s com $c = 12{,}5$ kg/s, e esse $c$ foi **ajustado** para dar uma velocidade terminal razoável. Em paraquedismo real, a velocidade terminal em queda livre de barriga é da ordem de 55 m/s, e a forma da curva $v(t)$ segue o arrasto quadrático, que se aproxima de $v_T$ mais depressa.

In [ ]:
c2 = m * g / v_exata(1e6) ** 2      # c_2 quadrático com a MESMA velocidade terminal do modelo linear
t_q, v_q = simular_salto_corrigido(0.01, 30.0, t_abertura=1e9, c2_queda_livre=c2)
t_fino = np.linspace(0, 30, 300)
dif = np.max(np.abs(np.interp(t_fino, t_q, v_q) - v_exata(t_fino)))
print(f"maior diferença entre os modelos: {numero(dif, 1)} m/s")

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_fino, y=v_exata(t_fino), name="arrasto linear", line=dict(color=CORES["aproximacao"])))
fig.add_trace(go.Scatter(x=t_q, y=v_q, name="arrasto quadrático", line=dict(color=CORES["secundaria"])))
fig.update_xaxes(title="tempo t (s)")
fig.update_yaxes(title="velocidade v (m/s)")
fig.update_layout(title="Mesma velocidade terminal, dinâmica diferente")
mostrar(fig)

**O que notar:** os dois modelos concordam no fim, mas diferem em mais de 7 m/s no meio da queda. Nenhuma verificação de código detecta isso. Só a **validação**, a comparação com medidas reais, diz qual modelo usar. Um agente pode escrever e até verificar o código, mas a escolha do modelo é uma decisão de engenharia.

## Resumo

| Conceito | Ideia central |
| --- | --- |
| Modelo matemático | uma lei física escrita como equação, com hipóteses explícitas |
| Discretização | troca do contínuo por passos $\Delta t$ e da derivada por uma diferença |
| Método de Euler | $v_{i+1} = v_i + f(t_i, v_i)\,\Delta t$, ordem 1 |
| Verificação | o código resolve o modelo corretamente? (solução exata, ordem medida) |
| Validação | o modelo representa a realidade? (comparação com medidas) |
| Estabilidade | passo grande demais pode fazer a solução numérica explodir |

## Exercícios

**Conceituais (sem computador)**

1. Mostre que {eq}`eq-paraquedista-exata` satisfaz {eq}`eq-paraquedista` e a condição inicial.
2. Para o modelo quadrático, deduza a velocidade terminal $v_T = \sqrt{mg/c_2}$ sem resolver a EDO.
3. Faça a análise de estabilidade da seção 7.1 para o modelo quadrático, linearizando $f(v) = g - (c_2/m)v^2$ em torno de $v_T$. Qual é o maior passo estável?

**Verificação (com agente)**

4. Peça a um agente uma simulação do salto com a densidade do ar variando com a altitude, $\rho(h) = \rho_0 e^{-h/H}$, com $\rho_0 \approx 1{,}2$ kg/m³ ao nível do mar e $H \approx 8{,}5$ km. Como $c_2 = \tfrac12 \rho C_D A$ (seção 6.1), o coeficiente passa a ser $c_2(h) = c_2^{(0)}\, \rho(h)/\rho_0$. Verifique o código, mostrando que ele reproduz o caso de densidade constante quando $H \to \infty$, e meça a ordem de convergência.
5. Peça a um agente que modele o esvaziamento de um tanque pela lei de Torricelli, $dh/dt = -k\sqrt{h}$. Compare com a solução exata e procure o que acontece quando $h$ chega a zero.

**Projeto aberto**

6. **Salto da estratosfera.** Em 2012, um paraquedista saltou de cerca de 39 km de altitude e ultrapassou a velocidade do som. Monte um modelo com gravidade e densidade variáveis, simule o salto e discuta quais hipóteses do modelo deste capítulo deixam de valer.

## Leituras

- S. C. Chapra e R. P. Canale, *Métodos Numéricos para Engenharia*, capítulo 1 (modelagem matemática).
- W. L. Oberkampf e C. J. Roy, *Verification and Validation in Scientific Computing*, Cambridge University Press, 2010, capítulos 1 e 2.